# PADV CA3 | Hospital Appointment No-Show Analytics
**Interactive Jupyter Notebook dashboard + Decision Tree classifier**  
Run cells in order. Keep `KaggleV2-May-2016.csv` beside this notebook, or select it when prompted. Charts use Plotly; controls use ipywidgets. No Streamlit required.

In [1]:

from google.colab import output
output.enable_custom_widget_manager()

import ipywidgets as widgets
import plotly.graph_objects as go

print("Interactive dashboard support enabled.")


Interactive dashboard support enabled.


In [2]:
# Run this cell once if any imports fail, then restart kernel and run all cells.
%pip install pandas numpy plotly scikit-learn ipywidgets nbformat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 33.1 MB/s  0:00:00


In [3]:

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from IPython.display import display, HTML
import ipywidgets as widgets

from google.colab import files, output
output.enable_custom_widget_manager()

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Load dataset in Google Colab
path = Path("KaggleV2-May-2016.csv")

if not path.exists():
    print("Please upload KaggleV2-May-2016.csv")
    uploaded = files.upload()

    if not uploaded:
        raise FileNotFoundError("No CSV file uploaded.")

    path = Path(next(iter(uploaded)))

df = pd.read_csv(path)

print("Dataset loaded successfully!")
print("Original rows / columns:", df.shape)

display(df.head())


Please upload KaggleV2-May-2016.csv


Saving KaggleV2-May-2016.csv to KaggleV2-May-2016.csv
Dataset loaded successfully!
Original rows / columns: (110527, 14)


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


In [4]:
required=['AppointmentID','Age','Gender','ScheduledDay','AppointmentDay','No-show','SMS_received','Hipertension','Diabetes','Alcoholism','Handcap','Scholarship']
missing=[c for c in required if c not in df.columns]
if missing: raise ValueError(f'Missing CSV columns: {missing}')
d=df.drop_duplicates(subset='AppointmentID').dropna(subset=required).copy()
d=d[d.Age.between(0,115) & d['No-show'].isin(['Yes','No']) & d.Gender.isin(['F','M'])].copy()
for c in ['ScheduledDay','AppointmentDay']:
    d[c]=pd.to_datetime(d[c],utc=True,errors='coerce')
d=d.dropna(subset=['ScheduledDay','AppointmentDay'])
d['WaitingDays']=(d.AppointmentDay.dt.normalize()-d.ScheduledDay.dt.normalize()).dt.days
d=d[d.WaitingDays>=0].copy()
d['NoShow']=d['No-show'].eq('Yes').astype(int)
d['AgeGroup']=pd.cut(d.Age,[-1,18,35,60,115],labels=['0-18','19-35','36-60','Above 60'])
d['Weekday']=d.AppointmentDay.dt.day_name()
d['AppointmentMonth']=d.AppointmentDay.dt.strftime('%Y-%m')
d['SMS Label']=d.SMS_received.map({0:'No SMS',1:'SMS Received'})
print('Cleaned rows:',len(d),'Removed:',len(df)-len(d),'No-show rate:',f'{d.NoShow.mean():.2%}')
d.to_csv('Cleaned_Hospital_Appointments.csv',index=False)


Cleaned rows: 110521 Removed: 6 No-show rate: 20.19%


## Interactive analytics dashboard
Change sidebar-style controls below; all charts update. Hover over bars, zoom and select using Plotly toolbar.

In [5]:

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Use the existing loaded dataset
d = df.copy()

d["Age"] = pd.to_numeric(d["Age"], errors="coerce")
d = d[d["Age"].between(0, 115)].copy()

d["NoShow"] = (
    d["No-show"].astype(str).str.strip().str.lower() == "yes"
).astype(int)

d["AgeGroup"] = pd.cut(
    d["Age"],
    [-1, 18, 35, 60, 115],
    labels=["0-18", "19-35", "36-60", "Above 60"]
)

d["ScheduledDay"] = pd.to_datetime(
    d["ScheduledDay"], utc=True, errors="coerce"
)
d["AppointmentDay"] = pd.to_datetime(
    d["AppointmentDay"], utc=True, errors="coerce"
)

d["WaitingDays"] = (
    d["AppointmentDay"].dt.normalize()
    - d["ScheduledDay"].dt.normalize()
).dt.days

d["Weekday"] = d["AppointmentDay"].dt.day_name()

d["WaitingGroup"] = pd.cut(
    d["WaitingDays"],
    [-1, 0, 3, 7, 14, 30, np.inf],
    labels=["Same day", "1-3", "4-7", "8-14", "15-30", "31+"]
)

# Filters
age = widgets.Dropdown(
    options=["All", "0-18", "19-35", "36-60", "Above 60"],
    description="Age:",
    layout=widgets.Layout(width="220px")
)

gender = widgets.Dropdown(
    options=["All", "F", "M"],
    description="Gender:",
    layout=widgets.Layout(width="220px")
)

sms = widgets.Dropdown(
    options=[("All", "All"), ("Received", 1), ("Not received", 0)],
    description="SMS:",
    layout=widgets.Layout(width="220px")
)

filters = widgets.HBox(
    [age, gender, sms],
    layout=widgets.Layout(
        flex_flow="row wrap",
        gap="12px",
        width="100%"
    )
)

out = widgets.Output()

def draw_dashboard(change=None):
    x = d.copy()

    if age.value != "All":
        x = x[x["AgeGroup"].astype(str) == age.value]
    if gender.value != "All":
        x = x[x["Gender"] == gender.value]
    if sms.value != "All":
        x = x[x["SMS_received"] == sms.value]

    with out:
        clear_output(wait=True)

        if x.empty:
            print("No matching appointments.")
            return

        total = len(x)
        missed = int(x["NoShow"].sum())
        attended = total - missed

        display(HTML(f"""
        <div style="background:#0b1220;color:white;
                    padding:20px;border-radius:14px;
                    font-family:Arial;max-width:1200px">
          <h2 style="color:#38bdf8;margin-top:0">
            HOSPITAL APPOINTMENT INTELLIGENCE
          </h2>
          <div style="display:grid;
                      grid-template-columns:repeat(4,1fr);
                      gap:12px">
            <div style="background:#172337;padding:15px;border-radius:10px">
              Appointments<h2>{total:,}</h2>
            </div>
            <div style="background:#172337;padding:15px;border-radius:10px">
              Attended<h2>{attended:,}</h2>
            </div>
            <div style="background:#172337;padding:15px;border-radius:10px">
              Missed<h2>{missed:,}</h2>
            </div>
            <div style="background:#172337;padding:15px;border-radius:10px">
              No-show Rate<h2>{missed/total*100:.2f}%</h2>
            </div>
          </div>
        </div>
        """))

        fig = make_subplots(
            rows=3,
            cols=2,
            specs=[
                [{"type": "domain"}, {"type": "xy"}],
                [{"type": "xy"}, {"type": "xy"}],
                [{"type": "xy"}, {"type": "xy"}]
            ],
            subplot_titles=[
                "Attendance Breakdown",
                "Age Group No-Show Rate",
                "No-Show by Waiting Time",
                "Medical Conditions",
                "Appointment Weekday Analysis",
                "SMS Reminder Comparison"
            ],
            vertical_spacing=0.12,
            horizontal_spacing=0.13
        )

        fig.add_trace(
            go.Pie(
                labels=["Attended", "Missed"],
                values=[attended, missed],
                hole=0.6,
                marker_colors=["#38bdf8", "#fb923c"],
                textinfo="percent",
                showlegend=False
            ),
            row=1, col=1
        )

        age_rate = (
            x.groupby("AgeGroup", observed=False)["NoShow"]
            .mean().mul(100)
        )
        fig.add_trace(
            go.Bar(
                x=age_rate.index.astype(str),
                y=age_rate.values,
                marker_color="#a78bfa"
            ),
            row=1, col=2
        )

        wait_rate = (
            x.groupby("WaitingGroup", observed=False)["NoShow"]
            .mean().mul(100)
        )
        fig.add_trace(
            go.Bar(
                x=wait_rate.index.astype(str),
                y=wait_rate.values,
                marker_color="#38bdf8"
            ),
            row=2, col=1
        )

        cond_names = ["Diabetes", "Hypertension", "Alcoholism"]
        cond_cols = ["Diabetes", "Hipertension", "Alcoholism"]
        cond_rates = [
            x.loc[x[c] == 1, "NoShow"].mean() * 100
            for c in cond_cols
        ]

        fig.add_trace(
            go.Bar(
                x=cond_names,
                y=cond_rates,
                marker_color="#fb923c"
            ),
            row=2, col=2
        )

        weekdays = [
            "Monday", "Tuesday", "Wednesday",
            "Thursday", "Friday", "Saturday", "Sunday"
        ]
        week_rate = (
            x.groupby("Weekday")["NoShow"]
            .mean().mul(100).reindex(weekdays)
        )
        fig.add_trace(
            go.Bar(
                x=week_rate.index,
                y=week_rate.values,
                marker_color="#34d399"
            ),
            row=3, col=1
        )

        sms_rate = (
            x.groupby("SMS_received")["NoShow"]
            .mean().mul(100).reindex([0, 1])
        )
        fig.add_trace(
            go.Bar(
                x=["Not Received", "Received"],
                y=sms_rate.values,
                marker_color=["#38bdf8", "#a78bfa"]
            ),
            row=3, col=2
        )

        fig.update_layout(
            template="plotly_dark",
            paper_bgcolor="#0b1220",
            plot_bgcolor="#0b1220",
            font=dict(color="#e5e7eb", size=12),
            height=1150,
            width=1100,
            showlegend=False,
            margin=dict(l=65, r=35, t=85, b=85),
            bargap=0.22
        )

        fig.update_yaxes(
            gridcolor="#334155",
            title_text="No-show rate (%)"
        )

        fig.update_xaxes(showgrid=False)

        fig.show(
            renderer="colab",
            config={"responsive": True, "displayModeBar": False}
        )

for control in [age, gender, sms]:
    control.observe(draw_dashboard, names="value")

display(HTML("<h3>Interactive Hospital Analytics Dashboard</h3>"))
display(filters)
display(out)

draw_dashboard()


Output()

## Decision Tree model and evaluation
Target is **NoShow = 1**. Features available before the appointment are used; patient identifiers and target are excluded. The 80/20 split is stratified. The metrics below are calculated from actual predictions, not fabricated.

In [6]:
feature_cols=['Age','Gender','Scholarship','Hipertension','Diabetes','Alcoholism','Handcap','SMS_received','WaitingDays','Weekday']
X=d[feature_cols].copy();y=d.NoShow
numeric=[c for c in feature_cols if c not in ['Gender','Weekday']]
pre=ColumnTransformer([('categories',OneHotEncoder(handle_unknown='ignore'),['Gender','Weekday']),('numbers','passthrough',numeric)])
model=Pipeline([('preprocess',pre),('classifier',DecisionTreeClassifier(max_depth=8,min_samples_leaf=50,class_weight='balanced',random_state=42))])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
model.fit(X_train,y_train);pred=model.predict(X_test)
metrics={'Accuracy':accuracy_score(y_test,pred),'Precision':precision_score(y_test,pred,zero_division=0),'Recall':recall_score(y_test,pred,zero_division=0),'F1 Score':f1_score(y_test,pred,zero_division=0)}
display(pd.DataFrame({'Metric':list(metrics),'Value':list(metrics.values())}).style.format({'Value':'{:.4f}'}))
print(classification_report(y_test,pred,target_names=['Attended','Missed'],zero_division=0))
cm=confusion_matrix(y_test,pred,labels=[0,1]);fig=px.imshow(cm,text_auto=True,x=['Attended','Missed'],y=['Attended','Missed'],color_continuous_scale='Blues',title='Confusion Matrix (Actual vs Predicted)',labels={'x':'Predicted','y':'Actual'});fig.show()
features=model.named_steps['preprocess'].get_feature_names_out();imp=pd.DataFrame({'Feature':features,'Importance':model.named_steps['classifier'].feature_importances_}).nlargest(12,'Importance')
px.bar(imp.sort_values('Importance'),x='Importance',y='Feature',orientation='h',template='plotly_dark',title='Top Prediction Features').show()
pd.DataFrame([metrics]).to_csv('Model_Metrics.csv',index=False)
print('Saved: Cleaned_Hospital_Appointments.csv and Model_Metrics.csv')

,Metric,Value
0,Accuracy,0.5882
1,Precision,0.3044
2,Recall,0.8085
3,F1 Score,0.4423


              precision    recall  f1-score   support

    Attended       0.92      0.53      0.67     17642
      Missed       0.30      0.81      0.44      4464

    accuracy                           0.59     22106
   macro avg       0.61      0.67      0.56     22106
weighted avg       0.79      0.59      0.63     22106



Saved: Cleaned_Hospital_Appointments.csv and Model_Metrics.csv


## Single appointment risk simulator
Illustrative decision support only, **not** a clinically validated system. Prediction depends on the dataset and the fitted model.

In [7]:
age_in=widgets.IntSlider(value=30,min=0,max=110,description='Age')
wait_in=widgets.IntSlider(value=7,min=0,max=120,description='Wait days')
gender_in=widgets.Dropdown(options=['F','M'],description='Gender')
sms_in=widgets.Dropdown(options=[0,1],description='SMS')
weekday_in=widgets.Dropdown(options=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday'],description='Weekday')
diabetes_in=widgets.Checkbox(description='Diabetes')
hyper_in=widgets.Checkbox(description='Hypertension')
btn=widgets.Button(description='Predict no-show risk',button_style='info');risk_out=widgets.Output()
def predict_click(_):
    row=pd.DataFrame([{'Age':age_in.value,'Gender':gender_in.value,'Scholarship':0,'Hipertension':int(hyper_in.value),'Diabetes':int(diabetes_in.value),'Alcoholism':0,'Handcap':0,'SMS_received':sms_in.value,'WaitingDays':wait_in.value,'Weekday':weekday_in.value}]);p=model.predict_proba(row)[0,1]
    with risk_out:risk_out.clear_output();print(f'Estimated no-show probability: {p:.1%} | Model prediction: {"Missed" if model.predict(row)[0] else "Attended"}')
btn.on_click(predict_click)
display(widgets.VBox([age_in,wait_in,gender_in,sms_in,weekday_in,diabetes_in,hyper_in,btn,risk_out]))